# AIRR Backend Parity Audit

Validate that germlines backend produces identical results to G3 backend.

In [ ]:
import os
import pandas as pd
from sadie.airr import Airr

In [ ]:
df = pd.read_csv('20260112_HCV_DB_example.csv')
print(f"Loaded {len(df)} sequences")
df.head()

In [ ]:
sequences = df[['sequence_id_heavy', 'sequence_heavy']].dropna()
sequences = sequences.rename(columns={'sequence_id_heavy': 'sequence_id', 'sequence_heavy': 'sequence'})
print(f"Prepared {len(sequences)} sequences for annotation")
sequences.head()

## Run with Germlines Backend

In [ ]:
os.environ['SADIE_USE_GERMLINES_MODULE'] = 'true'
airr_germlines = Airr('human')
result_germlines = airr_germlines.run_dataframe(sequences, seq_id_field='sequence_id', seq_field='sequence')
if 'source' in result_germlines.columns:
    result_germlines = result_germlines.drop(columns=['source'])
print(f"Germlines backend: {len(result_germlines)} results, {len(result_germlines.columns)} columns")
result_germlines.head()

## Run with G3 Backend

In [ ]:
os.environ['SADIE_USE_GERMLINES_MODULE'] = 'false'
airr_g3 = Airr('human')
result_g3 = airr_g3.run_dataframe(sequences, seq_id_field='sequence_id', seq_field='sequence')
if 'source' in result_g3.columns:
    result_g3 = result_g3.drop(columns=['source'])
print(f"G3 backend: {len(result_g3)} results, {len(result_g3.columns)} columns")
result_g3.head()

## Compare Results

In [ ]:
germlines_cols = set(result_germlines.columns)
g3_cols = set(result_g3.columns)

print(f"Germlines columns: {len(germlines_cols)}")
print(f"G3 columns: {len(g3_cols)}")
print(f"\nOnly in germlines: {germlines_cols - g3_cols}")
print(f"Only in G3: {g3_cols - germlines_cols}")
print(f"Common columns: {len(germlines_cols & g3_cols)}")

In [ ]:
common_cols = sorted(germlines_cols & g3_cols)
result_germlines_aligned = result_germlines[common_cols].sort_values('sequence_id').reset_index(drop=True)
result_g3_aligned = result_g3[common_cols].sort_values('sequence_id').reset_index(drop=True)

In [ ]:
differences = {}
for col in common_cols:
    mask = result_germlines_aligned[col].fillna('').astype(str) != result_g3_aligned[col].fillna('').astype(str)
    diff_count = mask.sum()
    if diff_count > 0:
        differences[col] = diff_count

if differences:
    print(f"Found differences in {len(differences)} columns:")
    for col, count in sorted(differences.items(), key=lambda x: -x[1]):
        print(f"  {col}: {count} differences")
else:
    print("No differences found - backends produce identical results")

In [ ]:
if differences:
    for col in list(differences.keys())[:5]:
        print(f"\n=== {col} ===")
        mask = result_germlines_aligned[col].fillna('').astype(str) != result_g3_aligned[col].fillna('').astype(str)
        sample_idx = mask[mask].index[:3]
        for idx in sample_idx:
            seq_id = result_germlines_aligned.loc[idx, 'sequence_id']
            print(f"  Sequence: {seq_id}")
            print(f"    Germlines: {result_germlines_aligned.loc[idx, col]}")
            print(f"    G3:        {result_g3_aligned.loc[idx, col]}")

## Summary

In [ ]:
total_values = len(common_cols) * len(result_germlines_aligned)
total_diffs = sum(differences.values()) if differences else 0
parity_pct = (1 - total_diffs / total_values) * 100 if total_values > 0 else 100

print("=" * 50)
print("AUDIT SUMMARY")
print("=" * 50)
print(f"Sequences tested: {len(sequences)}")
print(f"Common columns: {len(common_cols)}")
print(f"Total values compared: {total_values}")
print(f"Values with differences: {total_diffs}")
print(f"Parity: {parity_pct:.2f}%")
print("=" * 50)

if parity_pct == 100:
    print("PASS: Backends produce identical results")
else:
    print(f"FAIL: {len(differences)} columns have differences")